**Author:** Adrian Vanyi

 # Fetching S&P 500 historical members

This notebook shows how we fetch the historical members of the S&P 500 index (described in §6.2 of the documentation).

In essence, we source constituents from the revision history of Wikipedia's web page *List of S&P 500 companies* (https://en.wikipedia.org/wiki/List_of_S%26P_500_companies) . 


 ### Notebook structure
 0. Setup
 1. Fetch the latest Wikipedia revision before a given date
 2. Parse the constituents table from that revision
 3. Build the membership table at the first business day of each month
 4. Membership count over time

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import modules.sp500_historical_members as sp500
import modules.backtest_calendar as bcal
from modules import logging_setup

logging_setup.configure(level="INFO", stream = sys.stdout)

## 1. Fetch the latest Wikipedia revision before a given date

In [3]:
date = pd.Timestamp('2010-07-29')

index = "SPX"

page = sp500.WIKIPEDIA_PAGES[index]

revision = sp500.get_revisions_metadata( page_title = page,
                                   rvstart = date,
                                   rvdir = 'older',
                                   rvlimit = 1, 
                                   verbose = True)

print(revision)

[{'revid': 375904273, 'parentid': 375283445, 'user': '199.197.135.216', 'anon': True, 'timestamp': '2010-07-28T12:52:02Z', 'comment': ''}]


## 2. Parse the constituents table from that revision

In [4]:
members, members_info_df =sp500.get_index_members_at(index = "SPX", date =  date)

In [5]:
members_info_df

,Company,SEC filings,GICS Sector
Ticker symbol,,,
A,Agilent Technologies Inc,reports,Information Technology
AA,Alcoa Inc,reports,Materials
AAPL,Apple Inc.,reports,Information Technology
ABC,AmerisourceBergen Corp,reports,Health Care
ABT,Abbott Laboratories,reports,Health Care
...,...,...,...
XRX,Xerox Corp.,reports,Information Technology
YHOO,Yahoo Inc.,reports,Information Technology
YUM,Yum! Brands Inc,reports,Consumer Discretionary


##  3. Build the membership table at the first trading day of every month

In [6]:
start = pd.Timestamp("2007-04-06")
end = pd.Timestamp("2026-03-02")
first_trading_day_of_every_month = bcal.first_trading_day_of_every_month(start, end)

first_trading_day_of_month_to_sp500_members_dict = sp500.get_index_members_history(index = 'SPX', dates = first_trading_day_of_every_month)

[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-05-01 (n=500)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-06-01 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-07-02 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-08-01 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-09-04 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-10-01 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-11-01 (n=499)
[INFO modules.sp500_historical_members, get_index_members_history:296]  fetched index members for 2007-12-03 (n=498)
[INFO modules.sp500_historical_members, get_index_members_histor

In [7]:
sp500_members_at_start_of_months = pd.DataFrame.from_dict(first_trading_day_of_month_to_sp500_members_dict, orient = 'index')
sp500_members_at_start_of_months.index.rename("date", inplace=True)
sp500_members_at_start_of_months.index = pd.to_datetime(sp500_members_at_start_of_months.index)
sp500_members_at_start_of_months

,0,1,2,3,4,5,6,7,8,9,...,495,496,497,498,499,500,501,502,503,504
date,,,,,,,,,,,,,,,,,,,,,
2007-05-01,A,AA,AAPL,ABC,ABI,ABK,ABT,ACE,ACS,ADBE,...,XTO,YHOO,YUM,ZION,ZMH,NaN,NaN,NaN,NaN,NaN
2007-06-01,A,AA,AAPL,ABC,ABI,ABK,ABT,ACE,ACS,ADBE,...,YHOO,YUM,ZION,ZMH,NaN,NaN,NaN,NaN,NaN,NaN
2007-07-02,A,AA,AAPL,ABC,ABI,ABK,ABT,ACE,ACS,ADBE,...,YHOO,YUM,ZION,ZMH,NaN,NaN,NaN,NaN,NaN,NaN
2007-08-01,A,AA,AAPL,ABC,ABI,ABK,ABT,ACE,ACS,ADBE,...,YHOO,YUM,ZION,ZMH,NaN,NaN,NaN,NaN,NaN,NaN
2007-09-04,A,AA,AAPL,ABC,ABI,ABK,ABT,ACE,ACS,ADBE,...,YHOO,YUM,ZION,ZMH,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-03,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,NaN,NaN
2025-12-01,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,NaN,NaN,NaN
2026-01-02,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,NaN,NaN


## 4. Membership count over time

In [8]:
S = [set(v) - {None} for v in sp500_members_at_start_of_months.values]
different_lenghts = []
    
for s in S:
    n = len(s)
    if n not in different_lenghts:
       different_lenghts.append(n) 

sorted(different_lenghts)

[497, 498, 499, 500, 501, 502, 503, 504, 505]